# New Notebook

Start writing here...

In [1]:
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage
import asyncio

In [1]:
async for message in query(
    prompt="Explain what is an authentication flow",
    options=ClaudeAgentOptions(max_turns=1, allowed_tools=["Read", "Grep"]),
):
    if isinstance(message, ResultMessage):
        print(message.result)

## Authentication Flow

An **authentication flow** is the series of steps a system uses to verify the identity of a user (or service) before granting access to protected resources.

---

### Core Concepts

| Term | Description |
|---|---|
| **Authentication (AuthN)** | *Who are you?* — Verifying identity |
| **Authorization (AuthZ)** | *What can you do?* — Granting permissions |
| **Credentials** | Proof of identity (password, token, biometric, etc.) |
| **Session / Token** | A temporary proof that authentication succeeded |

---

### A Basic Authentication Flow (Username + Password)

```
User                    Client App               Server / Auth Service
 │                          │                           │
 │── enters credentials ──► │                           │
 │                          │── POST /login ───────────►│
 │                          │   { email, password }     │
 │                          │                           │── validates credentials
 │                 

In [1]:
# `/Users/sgaseretto/computer-science/aiml/agentic_coding/vibecoding/codex_vibes/solveish/notebooks/istockphoto-1620075616-612x612.jpg`

async for message in query(
    # prompt="Can you describe what you can see in istockphoto-1620075616-612x612.jpg",
    prompt="Can you describe what you can see in /Users/sgaseretto/computer-science/aiml/agentic_coding/vibecoding/codex_vibes/solveish/notebooks/istockphoto-1620075616-612x612.jpg",
    options=ClaudeAgentOptions(max_turns=2, allowed_tools=["Read", "Grep"]),
):
    if isinstance(message, ResultMessage):
        print(message.result)

The image is a **close-up macro photograph of a red ant** (likely a *fire ant* or *weaver ant*) perched on a **green plant stem**. Here are the details:

- **Subject**: A single red/orange ant, shown in great detail with its segmented body, long legs, and antennae clearly visible.
- **Pose**: The ant appears to be in an alert or active stance, with its front legs slightly raised and antennae extended forward.
- **Surface**: It's standing on a slender green stem or branch, with what appears to be a bud or node visible at the base.
- **Background**: The background is a **soft, blurred green bokeh**, suggesting a lush, leafy environment — typical of a shallow depth-of-field macro shot.
- **Lighting**: The image is well-lit, with natural-looking light that highlights the ant's reddish-orange coloring.
- **Style**: This appears to be a **stock photo** (consistent with the iStock filename), professionally composed for clarity and visual impact.

Overall, it's a striking macro nature photogra

Test passing the image directly to `query()`

In [1]:
import asyncio
import base64
from pathlib import Path
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock, ResultMessage


def load_image_as_base64(image_path: str) -> tuple[str, str]:
    """Load an image file and return (base64_data, media_type)."""
    path = Path(image_path)
    suffix = path.suffix.lower()

    media_type_map = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }

    media_type = media_type_map.get(suffix, "image/jpeg")
    with open(image_path, "rb") as f:
        data = base64.standard_b64encode(f.read()).decode("utf-8")

    return data, media_type


async def image_stream(image_path: str, question: str = "What do you see in this image?"):
    """Async generator that yields a user message with an image and a text prompt."""
    image_data, media_type = load_image_as_base64(image_path)

    yield {
        "type": "user",
        "message": {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": media_type,
                        "data": image_data,
                    },
                },
                {
                    "type": "text",
                    "text": question,
                },
            ],
        },
    }


async def describe_image(image_path: str, question: str = "What do you see in this image?"):
    """Use claude-agent-sdk's query() to describe an image."""
    options = ClaudeAgentOptions(
        system_prompt="You are a helpful assistant with excellent vision. Describe images clearly and in detail.",
    )

    print(f"Analyzing image: {image_path}\n")

    async for message in query(prompt=image_stream(image_path, question), options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)
        elif isinstance(message, ResultMessage):
            print(f"\n[Done — {message.num_turns} turn(s), cost: ${message.total_cost_usd:.4f}]")



# Change this to your image path
# IMAGE_PATH = "/Users/sgaseretto/computer-science/aiml/agentic_coding/vibecoding/codex_vibes/solveish/notebooks/istockphoto-1620075616-612x612.jpg"
# IMAGE_PATH = "istockphoto-1850729448-612x612-monarch.jpg"
IMAGE_PATH = "/Users/sgaseretto/computer-science/aiml/agentic_coding/vibecoding/codex_vibes/solveish/notebooks/istockphoto-1850729448-612x612-monarch.jpg"
QUESTION = "What do you see in this image? Describe it in detail."

asyncio.run(describe_image(IMAGE_PATH, QUESTION))

Analyzing image: /Users/sgaseretto/computer-science/aiml/agentic_coding/vibecoding/codex_vibes/solveish/notebooks/istockphoto-1850729448-612x612-monarch.jpg


## Image Description

This is a stunning nature photograph featuring **two Monarch butterflies** (*Danaus plexippus*) resting on a cluster of **bright pink Phlox flowers**.

### The Butterflies
- Both butterflies are **Monarchs**, instantly recognizable by their iconic **vivid orange wings** with **bold black veining** and **black borders dotted with white spots**
- Their wings are spread open, displaying the full, beautiful pattern
- The two butterflies are positioned facing **opposite directions**, creating a mirror-like, symmetrical composition
- Their bodies are black with white spots, typical of the species

### The Flowers
- The butterflies are perched on **garden phlox** (*Phlox paniculata*), a common pollinator-friendly plant
- The flowers are a rich **magenta/hot pink** color with small, delicate five-petaled blooms clustered together
- Additional phlox flowers are visible in the **soft-focus background**, adding depth and color

### The Setting & Composition
- The background 


[Done — 1 turn(s), cost: $0.0130]

Test passing the image to `query()` in a conversation that already has a conversation history (one previous message)

In [1]:
import asyncio
from claude_agent_sdk import query, AssistantMessage, TextBlock, ResultMessage

async def main():
    async for message in query(prompt="Say hello"):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)
        elif isinstance(message, ResultMessage):
            print(f"Done — cost: ${message.total_cost_usd:.4f}")

asyncio.run(main())

Hello! 👋 How can I help you today?

Done — cost: $0.0063

In [1]:
import asyncio
import base64
from pathlib import Path
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock, ResultMessage


# ── CONFIG ─────────────────────────────────────────────────────────────────────
IMAGE_PATH = "/Users/sgaseretto/computer-science/aiml/agentic_coding/vibecoding/codex_vibes/solveish/notebooks/istockphoto-1850729448-612x612-monarch.jpg"
QUESTION   = "What do you see in this image? Describe it in detail."
# ───────────────────────────────────────────────────────────────────────────────


def load_image_as_base64(image_path: str) -> tuple[str, str]:
    suffix = Path(image_path).suffix.lower()
    media_type_map = {
        ".jpg":  "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png":  "image/png",
        ".gif":  "image/gif",
        ".webp": "image/webp",
    }
    media_type = media_type_map.get(suffix, "image/jpeg")
    with open(image_path, "rb") as f:
        data = base64.standard_b64encode(f.read()).decode("utf-8")
    return data, media_type


async def conversation_stream(image_path: str, question: str):
    image_data, media_type = load_image_as_base64(image_path)

    # Only user messages in the stream — assistant history goes in system prompt
    history = [
        {
            "type": "user",
            "message": {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {"type": "base64", "media_type": media_type, "data": image_data},
                    },
                    {"type": "text", "text": question},
                ],
            },
        },
    ]

    for message in history:
        yield message


async def main():
    # Fake the prior conversation turns inside the system prompt
    system_prompt = """You are a helpful assistant with excellent vision.
Describe images clearly and in detail.

The conversation so far:
User: My name is John Doe
Assistant: Nice to meet you John, in what can I help you?

Continue the conversation naturally from here, keeping in mind the user's name is John."""

    options = ClaudeAgentOptions(system_prompt=system_prompt)

    print("=" * 60)
    print("[Turn 1] User:      My name is John Doe")
    print("[Turn 2] Assistant: Nice to meet you John, in what can I help you?")
    print(f"[Turn 3] User:      {QUESTION}")
    print("=" * 60)
    print("[Turn 4] Assistant: ", end="", flush=True)

    async for message in query(
        prompt=conversation_stream(IMAGE_PATH, QUESTION),
        options=options,
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text, end="", flush=True)
        elif isinstance(message, ResultMessage):
            print(f"\n\n[Done — {message.num_turns} turn(s), cost: ${message.total_cost_usd:.4f}]")


asyncio.run(main())

[Turn 1] User:      My name is John Doe

[Turn 2] Assistant: Nice to meet you John, in what can I help you?

[Turn 3] User:      What do you see in this image? Describe it in detail.

[Turn 4] Assistant: 

What a beautiful image, John! Here's a detailed description:

## Two Monarch Butterflies on Phlox Flowers

### The Butterflies
The image features **two Monarch butterflies (*Danaus plexippus*)** resting side by side on a cluster of flowers. Their wings are spread open, displaying the iconic Monarch pattern:
- **Vibrant orange** wing panels
- **Bold black borders and veining** that divide the wings into distinct sections
- **White spots** dotting the black borders and wing tips
- The butterflies appear to be mirror images of each other, creating a symmetrical, almost heart-like composition

### The Flowers
The butterflies are perched on what appears to be **garden phlox (*Phlox paniculata*)**, characterized by:
- Clusters of small, five-petaled flowers
- A bright **magenta/hot pink** color
- Delicate, rounded petals

### The Background
- The background is a soft, **blurred green** (bokeh effect), suggesting a lush garden setting with abundant foliage
- Additional pink phlox blooms are v



[Done — 1 turn(s), cost: $0.0133]